# 🏆 PTCG AI Battle — Mega Lucario ex Agent (Native `cg.api` Rewrite v3.1.1)

**Run All → Submit.** Generates `deck.csv` + `main.py` + `submission.tar.gz`.

*Last Updated: July 29, 2026*

| Component | Detail |
|---|---|
| Architecture | Score-based multi-select action ranking engine with neural scaffolding |
| Parser | Native `cg.api.to_observation_class()` ground-truth parser |
| Database | Engine-native `cg.api.all_card_data()` metadata dictionary |
| Tactical Engine | `AttackPlan` pre-computation, prize-aware target scoring, game-winning KO detection |
| Mega Lucario Heuristics | Riolu → Mega Lucario ex priority, energy deficit attachment, Supporter > Item > Basic hierarchy, bench-aware retreat |
| Options & Telemetry | Options Framework macro-intents (`MACRO_INTENTS`) + per-turn `decision_entropy()` telemetry logged to `game_log.jsonl` |
| Neural Scaffolding | 84→64→32→1 MLP stub (`σ(0)=0.5` until trained). Retained & wired for logging. |
| Submission Package | `submission.tar.gz` containing `main.py` (root), `deck.csv` (root), `cg/` package (<197.7 MiB) |

---

## Heuristic Priority & Scoring Table

| Action / Situation | Integer Score | Tactical Rationale |
|---|---|---|
| **Game-Winning KO Attack** | **50,000** | Knockout claims final prize(s) or clears opponent bench to instantly win game |
| **Knockout Attack** | **10,000 + Prize Bonus** | KO target (+3,000 Mega ex, +2,000 ex, +1,000 Basic) |
| **Non-KO Best Damage Attack** | **8,500 + Prize Bonus + Dmg** | Maximize damage output against highest-value target |
| **Evolve Active Riolu → Mega Lucario ex** | **8,000** | Power up active 340 HP main attacker |
| **Evolve Bench Riolu → Mega Lucario ex** | **7,000** | Prepare secondary 340 HP attacker on bench |
| **Play Supporter (Draw/Search)** | **6,500** | Refill hand resources (Professor's Research, Boss's Orders, etc.) |
| **Play Search Item (Ultra Ball/Nest Ball)** | **5,500** | Fetch key Pokémon / evolutionary pieces from deck |
| **Attach Energy (Active Deficit = 1)** | **5,000** | Enables immediate attack execution on active turn |
| **Attach Energy (Bench Attacker Deficit = 1)** | **4,500** | Charge benched backup attacker for upcoming turns |
| **Bench-Aware Retreat** | **4,200** | Active HP < 40% AND healthy bench attacker ready |
| **Use Beneficial Ability** | **4,000** | Zero-cost resource generation / search |
| **Play Basic Pokémon to Bench** | **3,500** | Expand bench size when bench_len < bench_max |
| **Play General Item / Tool Card** | **2,000** | General utility item placement |
| **End Turn** | **100** | Fallback when no beneficial actions remain |
| **Unsafe Retreat** | **-1,000** | Penalized to prevent retreating into unready/weak bench |


## 1. Generate `deck.csv`


In [ ]:
%%writefile deck.csv
677
677
677
677
678
678
678
678
673
673
674
674
675
675
676
676
333
333
1086
1086
1086
1086
1121
1121
1121
1121
1142
1142
1142
1123
1123
1123
1117
1117
1182
1182
1208
1208
1227
1227
1252
1156
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6


## 2. Generate `main.py`


In [ ]:
%%writefile main.py
"""
PTCG AI Agent - Mega Lucario ex Neuro-Symbolic Hybrid (Kaggle Submission v3.1.1)
Native cg.api engine integration, score-based multi-select action ranking, competitive heuristics.
Last Updated: 2026-07-29 | Patch 5: Macro-Intents + Decision Entropy Telemetry
"""

import sys
import os
import csv
import json
import logging
import math
import random
import numpy as np
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union

import cg.api as cg
from cg.api import (
    to_observation_class,
    all_card_data,
    AreaType,
    OptionType,
    SelectType,
    CardType,
    EnergyType,
    SelectContext,
    Card,
    Pokemon,
    PlayerState,
    Option,
    Select,
    Observation,
    Attack,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("ptcg_agent")

# Global Card Database initialized via native cg.api
CARD_DB: Dict[int, Card] = {}


def get_card_db() -> Dict[int, Card]:
    global CARD_DB
    if not CARD_DB:
        CARD_DB = all_card_data()
    return CARD_DB


def _opt_index(opt: Any, fallback: int = 0) -> int:
    """Safely extract option index from typed Option or dict."""
    if hasattr(opt, "index") and opt.index is not None:
        return opt.index
    if isinstance(opt, dict) and "index" in opt and opt["index"] is not None:
        try:
            return int(opt["index"])
        except Exception:
            pass
    return fallback


def resolve_card_id(opt: Option, obs: Observation) -> Optional[int]:
    """
    Resolve card ID for an option safely across typed and dict structures.
    CARD, TOOL_CARD, ENERGY_CARD carry card_id directly on wire.
    PLAY / EVOLVE carry hand position in in_play_index or index.
    """
    cid = getattr(opt, "card_id", None)
    if cid is None and isinstance(opt, dict):
        cid = opt.get("card_id")
    if cid is not None:
        return cid

    opt_type = getattr(opt, "option_type", None)
    if opt_type is None and isinstance(opt, dict):
        opt_type = opt.get("type")

    opt_type_str = str(getattr(opt_type, "name", opt_type)).upper()

    if opt_type_str in ("PLAY", "EVOLVE", "7", "8") or opt_type in (OptionType.PLAY, OptionType.EVOLVE):
        in_play_idx = getattr(opt, "in_play_index", None)
        if in_play_idx is None and isinstance(opt, dict):
            in_play_idx = opt.get("in_play_index")

        idx = in_play_idx if in_play_idx is not None else _opt_index(opt, 0)
        hand = getattr(obs, "my_hand", []) or []
        if idx is not None and 0 <= idx < len(hand):
            return hand[idx]

    return None


@dataclass
class AttackPlan:
    my_active_card: Optional[Card] = None
    opp_active_card: Optional[Card] = None
    opp_active_hp: int = 999
    best_attack_idx: Optional[int] = None
    best_attack_damage: int = 0
    is_ko: bool = False
    is_game_winning_ko: bool = False
    target_prize_value: int = 1


def compute_effective_damage(attack: Attack, attacker_card: Optional[Card], defender_card: Optional[Card]) -> int:
    if not attack:
        return 0
    base_dmg = attack.damage
    if base_dmg <= 0 or not defender_card:
        return base_dmg

    atk_type = (attacker_card.element_type if attacker_card else "").upper()
    weakness = (defender_card.weakness or "").upper()
    resistance = (defender_card.resistance or "").upper()

    mult = 1
    if atk_type and atk_type in weakness:
        mult = 2

    damage = base_dmg * mult

    if atk_type and atk_type in resistance:
        if "-30" in resistance:
            damage = max(0, damage - 30)
        elif "-20" in resistance:
            damage = max(0, damage - 20)

    return damage


def compute_attack_plan(obs: Observation) -> AttackPlan:
    plan = AttackPlan()
    db = get_card_db()

    my_active = obs.my_active[0] if obs.my_active else None
    opp_active = obs.opp_active[0] if obs.opp_active else None

    if not my_active or not opp_active:
        return plan

    plan.my_active_card = db.get(my_active.id)
    plan.opp_active_card = db.get(opp_active.id)
    plan.opp_active_hp = opp_active.hp

    # Prize-aware target scoring: 3 for Mega ex, 2 for ex, 1 for basic
    if plan.opp_active_card:
        name_lower = plan.opp_active_card.name.lower()
        stage_lower = (plan.opp_active_card.stage or "").lower()
        if "mega" in name_lower or "mega" in stage_lower:
            plan.target_prize_value = 3
        elif " ex" in name_lower or "ex" in name_lower or "ex" in stage_lower:
            plan.target_prize_value = 2
        else:
            plan.target_prize_value = 1
    else:
        plan.target_prize_value = 1

    # Evaluate best attack damage
    if plan.my_active_card and plan.my_active_card.attacks:
        best_dmg = -1
        best_i = None
        for i, atk in enumerate(plan.my_active_card.attacks):
            dmg = compute_effective_damage(atk, plan.my_active_card, plan.opp_active_card)
            if dmg > best_dmg:
                best_dmg = dmg
                best_i = i
        plan.best_attack_damage = max(0, best_dmg)
        plan.best_attack_idx = best_i

    plan.is_ko = plan.best_attack_damage >= plan.opp_active_hp
    prizes_needed = obs.my_prize_count if obs.my_prize_count > 0 else 6
    plan.is_game_winning_ko = plan.is_ko and (plan.target_prize_value >= prizes_needed or len(obs.opp_bench) == 0)

    return plan


def score_option(opt: Option, obs: Observation, plan: AttackPlan) -> int:
    db = get_card_db()
    opt_type = opt.option_type
    opt_type_str = str(opt_type.name if hasattr(opt_type, "name") else opt_type)
    ctx = obs.select.context if obs.select else None
    ctx_str = str(ctx.name if hasattr(ctx, "name") else ctx)

    # Prize bonus calculation: +3,000 for 3-prize Mega ex, +2,000 for 2-prize ex, +1,000 for Basic
    prize_bonus = 3000 if plan.target_prize_value == 3 else (2000 if plan.target_prize_value == 2 else 1000)

    # 1. Non-MAIN contexts (Setup / Switch / Search / Forced)
    if ctx_str in ("SETUP_ACTIVE_POKEMON", "SETUP_BENCH_POKEMON"):
        cid = resolve_card_id(opt, obs)
        if cid in (677, 333):  # Riolu
            return 9000
        elif cid:
            c = db.get(cid)
            if c and c.hp >= 100:
                return 7000
        return 5000 - _opt_index(opt)

    if ctx_str in ("SWITCH", "TO_ACTIVE"):
        if opt.in_play_index is not None and 0 <= opt.in_play_index < len(obs.my_bench):
            b_pkmn = obs.my_bench[opt.in_play_index]
            return 8000 + b_pkmn.hp
        return 5000 - _opt_index(opt)

    if ctx_str in ("IS_FIRST", "MULLIGAN"):
        if opt_type_str == "YES" or opt_type == OptionType.YES:
            return 9000
        return 1000

    # 2. MAIN context decisions

    # ATTACK
    if opt_type_str == "ATTACK" or opt_type == OptionType.ATTACK:
        # Game-winning KO check -> 50,000 (highest priority score)
        if plan.is_game_winning_ko:
            return 50000
        if plan.is_ko:
            return 10000 + prize_bonus
        return 8500 + prize_bonus + min(plan.best_attack_damage, 1000)

    # EVOLVE (Riolu -> Mega Lucario ex evolution priority: +8,000 active / +7,000 bench)
    if opt_type_str == "EVOLVE" or opt_type == OptionType.EVOLVE:
        cid = resolve_card_id(opt, obs)
        if cid == 678:  # Mega Lucario ex
            if opt.in_play_area == AreaType.ACTIVE or str(opt.in_play_area) in ("ACTIVE", "4"):
                return 8000
            return 7000
        return 6000

    # PLAY (Supporter 6,500 > Search Item 5,500 > Basic 3,500)
    if opt_type_str == "PLAY" or opt_type == OptionType.PLAY:
        cid = resolve_card_id(opt, obs)
        if cid:
            c = db.get(cid)
            if c:
                stage = (c.stage or "").lower()
                if "supporter" in stage:
                    return 6500
                if "item" in stage or "tool" in stage:
                    return 5500
                if "basic" in stage and len(obs.my_bench) < 5:
                    return 3500
        return 2000

    # ATTACH (Active energy deficit = 1 gets +5,000 active / +4,500 bench)
    if opt_type_str == "ATTACH" or opt_type == OptionType.ATTACH:
        is_active = opt.in_play_area == AreaType.ACTIVE or str(opt.in_play_area) in ("ACTIVE", "4")
        if is_active:
            my_act = obs.my_active[0] if obs.my_active else None
            my_act_card = plan.my_active_card
            if my_act and my_act_card and my_act_card.attacks:
                needed = my_act_card.attacks[0].energy_count
                curr = len(my_act.energies)
                if needed - curr == 1:
                    return 5000  # Energy deficit = 1 active
            return 4000
        else:
            return 4500 if len(obs.my_bench) > 0 else 3000  # Bench attachment

    # ABILITY
    if opt_type_str == "ABILITY" or opt_type == OptionType.ABILITY:
        return 4000

    # RETREAT (Bench-aware retreat: +4,200 only if active HP < 40% and healthy bench attacker ready; -1,000 if unsafe)
    if opt_type_str == "RETREAT" or opt_type == OptionType.RETREAT:
        my_act = obs.my_active[0] if obs.my_active else None
        my_card = plan.my_active_card
        if my_act and my_card and my_card.hp > 0:
            hp_ratio = my_act.hp / float(my_card.hp)
            has_bench_ready = any(b.hp >= 80 for b in obs.my_bench)
            if hp_ratio < 0.4 and has_bench_ready:
                return 4200
        return -1000

    # YES / NO
    if opt_type_str == "YES" or opt_type == OptionType.YES:
        return 5000
    if opt_type_str == "NO" or opt_type == OptionType.NO:
        return 1000

    # CARD / NUMBER / TOOL_CARD / ENERGY_CARD selection
    if opt_type_str in ("CARD", "TOOL_CARD", "ENERGY_CARD", "ENERGY", "NUMBER"):
        cid = resolve_card_id(opt, obs)
        if cid:
            c = db.get(cid)
            if c:
                if cid == 678:
                    return 8000
                if cid in (677, 333):
                    return 7000
                if "Supporter" in c.stage:
                    return 6000
                if "Energy" in c.stage:
                    return 5000
        return 3000 - _opt_index(opt)

    # END turn
    if opt_type_str == "END" or opt_type == OptionType.END:
        return 100

    # Default fallback
    return 1000 - _opt_index(opt)


# =============================================================================
# PATCH 5: MACRO-INTENTS (Options Framework) + DECISION ENTROPY
# Sutton, Precup & Singh 1999 — relabels score_option() buckets into named
# intents with initiation conditions. Zero compute cost; pure report value.
# =============================================================================

MACRO_INTENTS = {
    "LETHAL":    {"initiation": "KO wins game",                        "scores": (50000, 50000)},
    "AGGRO_KO":  {"initiation": "KO is feasible",                      "scores": (10000, 13999)},
    "SETUP":     {"initiation": "setup context (active/bench select)",  "scores": (9000, 9000)},
    "ATTACK":    {"initiation": "no better option, attack available",   "scores": (8500, 9999)},
    "DEVELOP":   {"initiation": "bench < 3 or evolution available",     "scores": (7000, 8499)},
    "RESOURCE":  {"initiation": "supporter/item in hand",              "scores": (5500, 6999)},
    "STABILIZE": {"initiation": "HP < 40% and healthy bench ready",    "scores": (4200, 4200)},
    "POWER_UP":  {"initiation": "energy deficit on attacker",          "scores": (4000, 5499)},
    "PASS":      {"initiation": "nothing useful",                      "scores": (0, 999)},
}


def macro_intent_for_score(score: int) -> str:
    """Map a score_option() output to its named macro-intent."""
    for name, spec in MACRO_INTENTS.items():
        lo, hi = spec["scores"]
        if lo <= score <= hi:
            return name
    return "UNKNOWN"


def decision_entropy(scores: List[int], temperature: float = 1000.0) -> float:
    """H = -Σ p_i log p_i over softmax-normalized scores.
    Cheap per-turn 'how contested was this decision' signal for the
    Strategy report's explainability section."""
    if len(scores) <= 1:
        return 0.0
    s = np.array(scores, dtype=np.float64)
    s = s - s.max()  # numerical stability
    exp_s = np.exp(s / temperature)
    p = exp_s / exp_s.sum()
    return float(-np.sum(p * np.log(p + 1e-12)))


# =============================================================================
# NEURAL SCAFFOLDING (Preserved & Inert during Rule Decisions)
# =============================================================================

class StateEncoder:
    """
    Encodes Observation into an 84-dimensional feature vector.
    Currently returns np.zeros(84, dtype=np.float32).
    """

    def encode(self, obs: Observation) -> np.ndarray:
        return np.zeros(84, dtype=np.float32)


class NeuralWorker:
    """
    Pure NumPy MLP for state value estimation.
    Architecture: 84 -> 64 (ReLU) -> 32 (ReLU) -> 1 (Sigmoid).
    """

    def __init__(self):
        self.encoder = StateEncoder()
        rng = np.random.RandomState(42)
        self.w1 = rng.randn(84, 64).astype(np.float32) * 0.1
        self.b1 = np.zeros(64, dtype=np.float32)
        self.w2 = rng.randn(64, 32).astype(np.float32) * 0.1
        self.b2 = np.zeros(32, dtype=np.float32)
        self.w3 = rng.randn(32, 1).astype(np.float32) * 0.1
        self.b3 = np.zeros(1, dtype=np.float32)

    def score_state(self, obs: Observation) -> float:
        x = self.encoder.encode(obs)
        x = np.maximum(0, np.dot(x, self.w1) + self.b1)
        x = np.maximum(0, np.dot(x, self.w2) + self.b2)
        out = np.dot(x, self.w3) + self.b3
        return float(1.0 / (1.0 + math.exp(-float(out[0]))))


class GameLogger:
    """Logs (observation_raw, action_indices) pairs for offline training."""

    def __init__(self, path="game_log.jsonl"):
        self._f = None
        try:
            self._f = open(path, "a", encoding="utf-8")
        except Exception:
            pass

    def log(self, obs_raw, action_indices, entropy=None, intent=None):
        if not self._f:
            return
        try:
            record = {"o": obs_raw, "a": action_indices}
            if entropy is not None:
                record["entropy"] = round(entropy, 4)
            if intent is not None:
                record["intent"] = intent
            self._f.write(json.dumps(record) + "\n")
            self._f.flush()
        except Exception:
            pass


# =============================================================================
# HEURISTIC ENGINE
# =============================================================================

class HeuristicEngine:
    def __init__(self, deck_path: str = "deck.csv"):
        self.deck_path = deck_path
        self.deck = self._load_deck(deck_path)
        get_card_db()
        self.neural_worker = NeuralWorker()

    def _load_deck(self, path: str) -> List[int]:
        d = []
        try:
            with open(path) as f:
                d = [int(l.strip()) for l in f if l.strip().isdigit()]
        except Exception:
            pass
        if len(d) != 60:
            d = (d + [6] * 60)[:60]
        return d

    def get_deck(self) -> List[int]:
        return self.deck

    def choose(self, obs: Observation) -> List[int]:
        if not obs.select or not obs.select.options:
            return []

        options = obs.select.options
        if len(options) == 1:
            return [_opt_index(options[0], 0)]

        plan = compute_attack_plan(obs)

        # Score every legal option
        scored_opts = []
        for list_idx, opt in enumerate(options):
            score = score_option(opt, obs, plan)
            wire_idx = _opt_index(opt, list_idx)
            scored_opts.append((score, wire_idx))

        # Sort options descending by integer score
        scored_opts.sort(key=lambda x: x[0], reverse=True)

        max_c = obs.select.max_count if obs.select.max_count > 0 else 1
        top_indices = [idx for score, idx in scored_opts[:max_c]]

        return top_indices


# =============================================================================
# AGENT ENTRY POINT
# =============================================================================

_engine = None
_logger = None


def _get_engine():
    global _engine
    if _engine is None:
        _engine = HeuristicEngine("deck.csv")
    return _engine


def _get_logger():
    global _logger
    if _logger is None:
        _logger = GameLogger("game_log.jsonl")
    return _logger


def action(obs: Observation) -> List[int]:
    """Action ranking entry point accepting typed Observation or raw dict."""
    if isinstance(obs, dict):
        parsed = to_observation_class(obs)
    else:
        parsed = obs

    if parsed.select is None:
        return _get_engine().get_deck()

    return _get_engine().choose(parsed)


def agent(observation: Any, configuration: Any = None) -> List[int]:
    """Kaggle environment entry point."""
    try:
        obs = to_observation_class(observation)
        if obs.select is None or obs.is_setup_phase:
            return _get_engine().get_deck()

        actions = _get_engine().choose(obs)

        # Patch 5: log entropy + macro-intent alongside action indices
        try:
            entropy_val = None
            intent_val = None
            if obs.select and obs.select.options:
                plan = compute_attack_plan(obs)
                scores = [score_option(opt, obs, plan) for opt in obs.select.options]
                entropy_val = decision_entropy(scores)
                if actions:
                    # Intent of the chosen action (highest-scored option)
                    top_score = max(scores)
                    intent_val = macro_intent_for_score(top_score)
            _get_logger().log(observation, actions, entropy=entropy_val, intent=intent_val)
        except Exception:
            pass
        return actions if actions else [0]
    except Exception as e:
        logger.error(f"Agent error: {e}")
        try:
            if observation and isinstance(observation, dict) and observation.get("select"):
                return [0]
            return _get_engine().get_deck()
        except Exception:
            return [0]


## 3. Package Submission (`submission.tar.gz`)
Creates the root-level submission archive containing `main.py`, `deck.csv`, and `cg/` package.


In [ ]:
import tarfile
import os

archive_path = "submission.tar.gz"

with tarfile.open(archive_path, "w:gz") as tar:
    tar.add("main.py", arcname="main.py")
    tar.add("deck.csv", arcname="deck.csv")
    if os.path.exists("cg"):
        tar.add("cg", arcname="cg")

assert os.path.exists(archive_path), "submission.tar.gz was not created!"
size_bytes = os.path.getsize(archive_path)
size_mb = size_bytes / (1024 * 1024)

print(f"📦 submission.tar.gz created successfully: {size_bytes:,} bytes ({size_mb:.2f} MiB)")
assert size_mb < 197.7, f"Submission size exceeds Kaggle 197.7 MiB limit: {size_mb:.2f} MiB"

with tarfile.open(archive_path, "r:gz") as tar:
    members = tar.getnames()
    print("Archive members (first 10):", members[:10])
    assert "main.py" in members, "main.py missing from archive root!"
    assert "deck.csv" in members, "deck.csv missing from archive root!"
    assert any(m.startswith("cg/") or m == "cg" for m in members), "cg/ missing from archive!"
print("✅ Submission packaging verification passed!")


## 4. Verification & Unit Tests
Executes comprehensive validation tests on `cg.api` observation parsing, game-winning KO detection, multi-select action ranking, submission archive structure, and `main.py` compilation.


In [ ]:
import sys
import os
import unittest
import importlib

# Ensure current directory is on sys.path
if "." not in sys.path:
    sys.path.insert(0, ".")

# Clear cached main module if already loaded
if "main" in sys.modules:
    del sys.modules["main"]

import main
import cg.api as cg
from cg.api import to_observation_class, OptionType, SelectType, SelectContext, AreaType

print("=== Running Verification Test Suite ===")

# Test 1: Verify deck loading & Setup Phase return
deck = main._get_engine().get_deck()
assert len(deck) == 60, f"Expected 60 deck cards, got {len(deck)}"
assert all(isinstance(c, int) for c in deck), "All deck cards must be integers"
print("✅ Test 1 Passed: Deck list contains 60 valid integer card IDs")

# Test 2: Verify to_observation_class parsing
mock_setup = {"current": {"yourIndex": 0, "players": [{"handCount": 7}, {"handCount": 7}]}, "select": None}
parsed_setup = to_observation_class(mock_setup)
assert parsed_setup.is_setup_phase is True
assert parsed_setup.select is None
setup_res = main.agent(mock_setup)
assert len(setup_res) == 60
print("✅ Test 2 Passed: to_observation_class handles Setup Phase (returns 60-card list)")

# Test 3: Game-Winning KO Detection (score = 50,000)
mock_lethal = {
    "current": {
        "yourIndex": 0,
        "players": [
            {
                "active": [{"id": 677, "serial": 1, "hp": 80, "maxHp": 80, "energies": [6]}],
                "bench": [],
                "hand": [{"id": 1182}],
                "prize": [1]
            },
            {
                "active": [{"id": 677, "serial": 2, "hp": 30, "maxHp": 80}],
                "bench": [],
                "prize": [1, 2, 3, 4, 5, 6]
            }
        ]
    },
    "select": {
        "type": "0",
        "context": "0",
        "option": [
            {"type": "7", "area": "2", "index": 0},
            {"type": "13", "area": "4", "index": 0, "attack_id": 0},
            {"type": "14", "area": "11", "index": 0}
        ],
        "minCount": 1,
        "maxCount": 1
    }
}
parsed_lethal = to_observation_class(mock_lethal)
plan = main.compute_attack_plan(parsed_lethal)
assert plan.is_ko is True, "Attack must be KO"
assert plan.is_game_winning_ko is True, "Attack must be game-winning KO"

attack_opt = parsed_lethal.select.options[1]
attack_score = main.score_option(attack_opt, parsed_lethal, plan)
assert attack_score == 50000, f"Expected game-winning KO score 50000, got {attack_score}"

action_res = main.agent(mock_lethal)
assert action_res == [1], f"Expected action [1], got {action_res}"
print("✅ Test 3 Passed: Game-winning KO detected with score 50,000 & prioritized over Supporter")

# Test 4: Multi-Select Handling (maxCount > 1)
mock_multi = {
    "current": {
        "yourIndex": 0,
        "players": [
            {
                "active": [{"id": 677, "hp": 80}],
                "hand": [{"id": 678}, {"id": 677}, {"id": 6}, {"id": 1086}],
                "prize": [1, 2, 3, 4, 5, 6]
            },
            {
                "active": [{"id": 678, "hp": 340}],
                "prize": [1, 2, 3, 4, 5, 6]
            }
        ]
    },
    "select": {
        "type": "1",
        "context": "8",
        "option": [
            {"type": "3", "card_id": 678, "index": 0},
            {"type": "3", "card_id": 677, "index": 1},
            {"type": "3", "card_id": 6, "index": 2},
            {"type": "3", "card_id": 1086, "index": 3}
        ],
        "minCount": 2,
        "maxCount": 2
    }
}
multi_res = main.agent(mock_multi)
assert len(multi_res) == 2, f"Expected 2 options for maxCount=2, got {len(multi_res)}"
assert multi_res == [0, 1], f"Expected top 2 indices [0, 1], got {multi_res}"
print("✅ Test 4 Passed: Multi-select ranking returns top maxCount indices [0, 1]")

# Test 5: Archive Structure and Size Verification
import tarfile
assert os.path.exists("submission.tar.gz"), "submission.tar.gz missing"
with tarfile.open("submission.tar.gz", "r:gz") as tar:
    names = tar.getnames()
    assert "main.py" in names, "main.py missing from archive"
    assert "deck.csv" in names, "deck.csv missing from archive"
    assert any(n.startswith("cg/") or n == "cg" for n in names), "cg/ directory missing from archive"
    sz_mb = os.path.getsize("submission.tar.gz") / (1024 * 1024)
    assert sz_mb < 197.7, f"Archive size {sz_mb:.2f} MiB exceeds 197.7 MiB"
print(f"✅ Test 5 Passed: submission.tar.gz valid ({sz_mb:.2f} MiB < 197.7 MiB limit)")

# Test 6: Verify Patch 5 (Macro-Intents & Decision Entropy Telemetry)
assert main.macro_intent_for_score(50000) == "LETHAL"
assert main.macro_intent_for_score(10000) == "AGGRO_KO"
assert main.macro_intent_for_score(9000) == "SETUP"
assert main.macro_intent_for_score(8500) == "ATTACK"
assert main.macro_intent_for_score(6500) == "RESOURCE"
assert main.macro_intent_for_score(4200) == "STABILIZE"
assert main.macro_intent_for_score(100) == "PASS"

e_uniform = main.decision_entropy([100, 100, 100, 100])
e_dominated = main.decision_entropy([50000, 100, 100, 100])
assert e_uniform > e_dominated
assert main.decision_entropy([]) == 0.0
assert main.decision_entropy([50000]) == 0.0
print("✅ Test 6 Passed: Macro-Intents mapping & Decision Entropy telemetry working")

print("
🎉 ALL VERIFICATION TESTS PASSED 100%!")
